In [1]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

# Evaluación
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

# Álgebra lineal
from scipy.linalg import cho_factor, cho_solve

# Utilidades
from pathlib import Path
from time import perf_counter
from math import comb

# Configuración
RANDOM_STATE = 42
EPSILON = 1e-6
PERCENTILES = [50, 75, 90, 95, 99]

DATA_DIR = Path("data")
RESULTS_DIR = Path("resultados")

RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [11]:
# Download latest version
path = Path(
    kagglehub.dataset_download(
        "mohankrishnathalla/credit-card-fraud-detection-dataset-2026"
    )
)

print("Path to dataset files:", path)

# Load dataset
df = pd.read_csv(
    path / "credit_card_fraud_detection_2026.csv"
)

display(df.head())

Path to dataset files: C:\Users\Practicante2\.cache\kagglehub\datasets\mohankrishnathalla\credit-card-fraud-detection-dataset-2026\versions\46


,transaction_id,customer_id,card_id,merchant_id,transaction_datetime,transaction_amount,merchant_category,merchant_country,transaction_hour,transaction_day_of_week,card_present,customer_age,customer_gender,customer_state,account_age_days,credit_limit,credit_score,num_cards_held,transactions_last_1h,transactions_last_24h,transactions_last_7d,amount_last_1h,amount_last_24h,days_since_last_transaction,distance_from_home_km,distance_from_last_tx_km,international_transaction,high_risk_country,amount_vs_avg_ratio,used_chip,used_pin,transaction_channel,device_type,browser,is_weekend,avg_transaction_amount_30d,std_transaction_amount_30d,num_declined_last_30d,num_chargebacks_ever,email_domain,is_fraud,fraud_type,fraud_amount_lost
0,TXN-00000001,CUS-00006042,CRD-00006042,MER-00002507,2026-01-01 02:28:51,37.36,Healthcare & Pharmacy,USA,2,Monday,0,41,Female,PA,709,6630.0,742,2,1,2,3,88.91,209.94,0.9,13.0,0.3,0,0,0.398,0,0,Online,Mobile,Chrome,0,93.91,39.48,0,0,corporate,0,Legitimate,0.0
1,TXN-00000002,CUS-00005262,CRD-00005262,MER-00000189,2026-01-01 03:15:18,50.04,Restaurant & Dining,Germany,3,Monday,1,56,Male,WV,1143,4350.0,602,1,6,6,23,1290.10,1093.31,6.6,5311.8,17.0,1,0,0.278,1,1,In-store,POS Terminal,NaN,0,180.05,134.45,0,0,yahoo,0,Legitimate,0.0
2,TXN-00000003,CUS-00005901,CRD-00005901,MER-00002054,2026-01-01 03:44:38,38.87,Grocery & Supermarket,Australia,3,Monday,0,43,Male,MN,601,3950.0,605,4,1,1,3,36.27,46.54,4.2,14164.5,53.2,1,0,0.925,0,0,Online,Desktop,Chrome,0,42.00,22.46,0,0,hotmail,0,Legitimate,0.0
3,TXN-00000004,CUS-00006958,CRD-00006958,MER-00000310,2026-01-01 03:53:41,46.60,Gas Station,China,3,Monday,0,51,Female,OH,582,13870.0,675,5,0,2,3,0.00,134.35,3.0,1119.1,3.6,1,1,0.718,0,1,ATM,ATM,NaN,0,64.87,29.76,0,0,gmail,0,Legitimate,0.0
4,TXN-00000005,CUS-00003543,CRD-00003543,MER-00001197,2026-01-01 04:29:08,62.13,Clothing & Apparel,UK,4,Monday,1,40,Female,PA,711,6570.0,743,2,1,2,2,38.37,117.74,0.9,6564.5,22.4,1,0,1.317,1,0,In-store,POS Terminal,NaN,0,47.19,18.52,0,0,gmail,0,Legitimate,0.0


### Fase 3. Adquisición y preparación de los datos

In [15]:
n, d = df.shape

print(f"Número de observaciones (n): {n:,}")
print(f"Número de columnas (d): {d}")
print(f"Registros duplicados: {df.duplicated().sum():,}")

Número de observaciones (n): 100,000
Número de columnas (d): 43
Registros duplicados: 0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   transaction_id               100000 non-null  object 
 1   customer_id                  100000 non-null  object 
 2   card_id                      100000 non-null  object 
 3   merchant_id                  100000 non-null  object 
 4   transaction_datetime         100000 non-null  object 
 5   transaction_amount           100000 non-null  float64
 6   merchant_category            100000 non-null  object 
 7   merchant_country             100000 non-null  object 
 8   transaction_hour             100000 non-null  int64  
 9   transaction_day_of_week      100000 non-null  object 
 10  card_present                 100000 non-null  int64  
 11  customer_age                 100000 non-null  int64  
 12  customer_gender              100000 non-null  object 
 13  

In [17]:
df_diagnostico = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "valores_nulos": df.isna().sum(),
    "porcentaje_nulos": df.isna().mean() * 100
})

display(df_diagnostico)

,tipo_dato,valores_unicos,valores_nulos,porcentaje_nulos
transaction_id,object,100000,0,0.000
customer_id,object,8000,0,0.000
card_id,object,8000,0,0.000
merchant_id,object,3000,0,0.000
transaction_datetime,object,99762,0,0.000
transaction_amount,float64,30862,0,0.000
merchant_category,object,15,0,0.000
merchant_country,object,20,0,0.000
transaction_hour,int64,24,0,0.000
transaction_day_of_week,object,7,0,0.000


#### Solo hay una columna con valores nulos y es 'browser'; es decir, de qué navegador se hizo la consulta. Es una variable categórica

In [18]:
display(df.describe().T)

,count,mean,std,min,25%,50%,75%,max
transaction_amount,100000.0,126.871977,276.180425,0.500,21.2800,52.260,129.72000,15000.000
transaction_hour,100000.0,14.393860,5.018253,0.000,11.0000,14.000,19.00000,23.000
card_present,100000.0,0.357770,0.479346,0.000,0.0000,0.000,1.00000,1.000
customer_age,100000.0,42.223670,14.246361,18.000,32.0000,42.000,52.00000,85.000
account_age_days,100000.0,785.451240,747.492434,1.000,233.0000,554.000,1103.00000,3650.000
credit_limit,100000.0,10083.182000,6510.646682,500.000,5000.0000,8500.000,13650.00000,36670.000
credit_score,100000.0,682.329280,92.122179,328.000,620.0000,683.000,748.00000,850.000
num_cards_held,100000.0,2.198830,1.163491,1.000,1.0000,2.000,3.00000,5.000
transactions_last_1h,100000.0,0.783890,1.348801,0.000,0.0000,0.000,1.00000,8.000
transactions_last_24h,100000.0,2.951760,2.382417,0.000,1.0000,2.000,4.00000,18.000


In [ ]:
# Seleccionar solo México de la lista de países
df["merchant_country"].value_counts()

merchant_country
USA             44515
UK               8349
Canada           7238
Germany          5651
Australia        5423
France           4521
Italy            3344
Netherlands      3039
Spain            2849
Brazil           2354
India            2175
Mexico           1921
China            1682
Nigeria          1261
South Africa     1140
Russia           1046
Romania           943
UAE               940
Japan             821
South Korea       788
Name: count, dtype: int64

In [20]:
df_mx = df[df["merchant_country"] == "Mexico"]

n, d = df_mx.shape

print(f"Transacciones en México: {n:,}")
print(f"Número de columnas: {d}")
print(f"Porcentaje del dataset original: {n / len(df):.2%}")

Transacciones en México: 1,921
Número de columnas: 43
Porcentaje del dataset original: 1.92%
